In [ ]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from deap import base, creator, tools, algorithms
import numpy as np
from scipy.spatial import cKDTree


In [ ]:
data_raw = pd.read_csv('data.csv')
data_raw.head()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display(data_raw)

In [ ]:
# Konstan
g = 9.81
p = 997  # Densitas air pada 27 °C
r = 0.11 / 2  # Radius poros luar turbin


In [ ]:
data = data_raw.dropna(subset=['AV']).copy()
data['Q'] = (data['Flowrate'] * 60000).round(1)
data['S'] = data['Screw']
data['H'] = data['Head']

data['Torsi'] = data['Mass'] * g * r
data['Daya'] = data['Torsi'] * data['AV']
data['Hidrolik'] = p * g * data['Flowrate'] * data['Head']
data['Efisiensi'] = data['Daya'] / data['Hidrolik']

n = data['Efisiensi'].dropna()


**Pengujian ANOVA**

In [ ]:
# ANOVA
anova_formula = 'n ~ C(S) + C(H) + C(Q) + C(S):C(Q) + C(S):C(H) + C(H):C(Q)'
anova_model = ols(anova_formula, data=data).fit()
anova_results = sm.stats.anova_lm(anova_model, typ=2)

print(anova_results)


Pengujian Tukey HSD

In [ ]:
data['SQ'] = data['S'].astype(str) + ' : ' + data['Q'].astype(str)

# Tukey HSD Test
tukey_terms = ['S', 'Q', 'SQ']

for term in tukey_terms:
    tukey = pairwise_tukeyhsd(endog=data['Efisiensi'], groups=data[term], alpha=0.05)
    if any(tukey.pvalues < 0.05):
        print(f"{term}:")
        print(tukey)


**Pengujian Algoritma Genetika**

In [ ]:
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0, 1.0))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_screw", np.random.choice, [1, 2])
toolbox.register("attr_head", np.random.uniform, data['Head'].min(), data['Head'].max())
toolbox.register("attr_flowrate", np.random.uniform, data['Flowrate'].min(), data['Flowrate'].max())

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_screw, toolbox.attr_head, toolbox.attr_flowrate), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Evaluasi Fitness
def evaluate_adjusted(individual):
    screw, head, flowrate = individual
    tree = cKDTree(data[['Head', 'Flowrate']])
    _, idx = tree.query([head, flowrate], k=1)
    nearest_row = data.iloc[idx]

    if nearest_row['Screw'] != screw:
        return 0, 0

    m = nearest_row['Mass']
    omega = nearest_row['AV']
    torque = m * g * r
    P = torque * omega
    hydraulic_P = p * g * flowrate * head
    efficiency = P / hydraulic_P if hydraulic_P > 0 else 0

    if efficiency > 1:
        return 0, 0
    return P, efficiency

toolbox.register("evaluate", evaluate_adjusted)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

# Eksekusi Algoritma
def run_ga_adjusted():
    pop = toolbox.population(n=100)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("daya_avg", lambda fits: np.mean([f[0] for f in fits]))
    stats.register("efisiensi_avg", lambda fits: np.mean([f[1] for f in fits]))
    stats.register("daya_max", lambda fits: max([f[0] for f in fits]))
    stats.register("efisiensi_max", lambda fits: max([f[1] for f in fits]))

    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + stats.fields

    pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=60, stats=stats, halloffame=hof, verbose=True)

    best_individual_adjusted = hof[0]
    efficiency_percent = "{:.2f}".format(best_individual_adjusted.fitness.values[1] * 100)  # Convert efficiency to percent
    print(f"Daya: {best_individual_adjusted.fitness.values[0]:.2f}, Efisiensi: {efficiency_percent} %")
    print(f"Screw: {best_individual_adjusted[0]}, Head: {best_individual_adjusted[1]:.2f}, Laju Aliran: {best_individual_adjusted[2]:.6f}")

    return best_individual_adjusted, logbook

best_individual_adjusted, logbook = run_ga_adjusted()


Pengujian Konvergensi

In [ ]:
eff = [log['efisiensi_max'] for log in logbook]

c = 0.001
k = 20

konvergen = None

for gen in range(k, len(eff)):
    if abs(eff[gen] - eff[gen - k]) <= c:
        all_subsequent_met = all(
            abs(eff[g] - eff[g - k]) <= c
            for g in range(gen, len(eff))
        )
        if all_subsequent_met:
            konvergen = gen - k
            break

konvergen
